In [1]:
import pandas as pd

#files loaded
poverty = pd.read_csv(r"C:\Users\pilla\OneDrive\Documents\Desktop\Capstone\Capstone Project Data\% of People in Poverty AZ City.csv")
single_parent = pd.read_csv(r"C:\Users\pilla\OneDrive\Documents\Desktop\Capstone\Capstone Project Data\% Single Parent Households by AZ City.csv")
unemployment = pd.read_csv(r"C:\Users\pilla\OneDrive\Documents\Desktop\Capstone\Capstone Project Data\% Unemployed AZ City.csv")
youth = pd.read_csv(r"C:\Users\pilla\OneDrive\Documents\Desktop\Capstone\Capstone Project Data\% Youth Enrolled in School AZ City.csv")
population = pd.read_csv(r"C:\Users\pilla\OneDrive\Documents\Desktop\Capstone\Capstone Project Data\AZ Population by City 2020.csv")
burglary = pd.read_csv(r"C:\Users\pilla\OneDrive\Documents\Desktop\Capstone\Capstone Project Data\Burglaries per 100k AZ City.csv")
vacancy = pd.read_csv(r"C:\Users\pilla\OneDrive\Documents\Desktop\Capstone\Capstone Project Data\Vacancy (%) by City AZ.csv")

#function to clean each file
def clean_file(df, value_col, new_name):
    #removed metadata row
    df = df.iloc[1:].copy()

    #kept only needed column names
    df = df[["Geography Name", "GeoID", value_col]].copy()

    #renamed columns
    df = df.rename(columns={
        "Geography Name": "City",
        value_col: new_name
    })

    #clean values
    df["City"] = df["City"].astype(str).str.strip()
    df["GeoID"] = pd.to_numeric(df["GeoID"], errors="coerce")
    df[new_name] = pd.to_numeric(df[new_name], errors="coerce")

    #removed duplicates
    df = df.drop_duplicates(subset=["City", "GeoID"])

    return df

#cleaned each dataset
poverty_clean = clean_file(
    poverty,
    "Percent of People in Poverty",
    "PovertyRate"
)

single_parent_clean = clean_file(
    single_parent,
    "Percent Single-Headed Household Living with Children",
    "SingleParentRate"
)

unemployment_clean = clean_file(
    unemployment,
    "Percent People Unemployed",
    "UnemploymentRate"
)

youth_clean = clean_file(
    youth,
    "Percent School Students Ages 3-17",
    "YouthEnrollmentRate"
)

population_clean = clean_file(
    population,
    "Population",
    "Population"
)

burglary_clean = clean_file(
    burglary,
    "Burglaries and Larcenies/100,000 People",
    "BurglaryRate"
)

vacancy_clean = clean_file(
    vacancy,
    "Vacancy Rate",
    "VacancyRate"
)

#merged together
merged = (
    poverty_clean
    .merge(single_parent_clean, on=["City", "GeoID"], how="inner")
    .merge(unemployment_clean, on=["City", "GeoID"], how="inner")
    .merge(youth_clean, on=["City", "GeoID"], how="inner")
    .merge(population_clean, on=["City", "GeoID"], how="inner")
    .merge(burglary_clean, on=["City", "GeoID"], how="inner")
    .merge(vacancy_clean, on=["City", "GeoID"], how="inner")
)

#removed missing values
merged = merged.dropna(subset=["BurglaryRate"])
merged = merged.dropna()

#filtered dataset for only cities with 100000 or more population
merged = merged[merged["Population"] > 100000].copy()

#reset index after filtering
merged.reset_index(drop=True, inplace=True)

#sorted and saved
merged = merged.sort_values("City").reset_index(drop=True)

merged.to_csv(
    r"C:\Users\pilla\OneDrive\Documents\Desktop\Capstone\Capstone Project Data\merged_cleaned_data.csv",
    index=False
)

#checked results
print(merged.head())
print("\nShape:", merged.shape)
print("\nCities kept:\n", merged["City"].tolist())
print("\nMissing values:\n", merged.isnull().sum())

       City   GeoID  PovertyRate  SingleParentRate  UnemploymentRate  \
0   Buckeye  407940         8.00              5.37              4.91   
1  Chandler  412000         7.93              7.19              3.62   
2   Gilbert  427400         5.26              6.67              3.50   
3  Glendale  427820        15.11              7.70              5.61   
4  Goodyear  428380         5.25              4.71              4.18   

   YouthEnrollmentRate  Population  BurglaryRate  VacancyRate  
0                72.35    104923.0        1307.2         7.39  
1                79.63    280136.0        1689.3         4.53  
2                80.55    280262.0        1009.1         3.96  
3                72.29    252833.0        2425.7         5.44  
4                75.13    107645.0        1929.5         9.42  

Shape: (13, 9)

Cities kept:
 ['Buckeye', 'Chandler', 'Gilbert', 'Glendale', 'Goodyear', 'Mesa', 'Peoria', 'Phoenix', 'Scottsdale', 'Surprise', 'Tempe', 'Tucson', 'Yuma']

Missing va